# Tariikhna — Production Data Pipeline

Multi-layer pipeline that converts long Sira passages into fine-tuning training data.

### Production features
- **Multi-API-key rotation**: assign different Groq keys to different layers
- **Rate-limit waiting**: when a key hits its limit, waits until the limit resets (does NOT skip work)
- **Full response logging**: every layer's raw output is saved for review
- **JSON file input**: loops over a file containing many long passages
- **Resumable**: saves progress after each passage, can restart without losing work
- **Auto-visualizations**: generates charts for your graduation document

### Pipeline layers
```
1a Split passage into narrative units
1b Select visually-worthy units
1c Segment units into scenes
2  Generate schema + 250-300w prompt   <- the layer you fine-tune
3  Safety filter (NSFW avoidance)
```

## 1. Setup & Install

In [ ]:
!pip install groq matplotlib -q

In [ ]:
import os
import json
import time
import re
from datetime import datetime
from groq import Groq

# Output directories
os.makedirs('pipeline_output', exist_ok=True)
os.makedirs('pipeline_output/logs', exist_ok=True)
os.makedirs('pipeline_output/schemas', exist_ok=True)
os.makedirs('pipeline_output/charts', exist_ok=True)
print('Directories ready!')

## 2. Multi-API-Key Configuration

Assign different Groq API keys to different layers so no single key gets overloaded.
You can use the same key for multiple layers, or give each its own. Add as many keys as you have.

In [ ]:
# ============================================
# PASTE YOUR GROQ API KEYS HERE
# Get keys free at: https://console.groq.com/keys
# You can list as many as you have - they rotate within each layer.
# ============================================

API_KEYS = {
    # Each layer gets a LIST of keys. They rotate when one hits a limit.
    'layer_1a': ['gsk_KEY_A_here'],
    'layer_1b': ['gsk_KEY_B_here'],
    'layer_1c': ['gsk_KEY_C_here'],
    'layer_2':  ['gsk_KEY_D_here'],   # the most-used layer - give it your best/most keys
    'layer_3':  ['gsk_KEY_E_here'],
}

# EXAMPLE with multiple keys per layer and sharing:
# API_KEYS = {
#     'layer_1a': ['gsk_key1'],
#     'layer_1b': ['gsk_key1'],          # 1a and 1b share key1 (they run less often)
#     'layer_1c': ['gsk_key2'],
#     'layer_2':  ['gsk_key3', 'gsk_key4'],  # layer 2 rotates between two keys
#     'layer_3':  ['gsk_key5'],
# }

MODEL = 'llama-3.1-8b-instant'

# Validate
all_keys = set()
for layer, keys in API_KEYS.items():
    for k in keys:
        all_keys.add(k)
print(f'Configured {len(all_keys)} unique key(s) across {len(API_KEYS)} layers')
for layer, keys in API_KEYS.items():
    print(f'  {layer}: {len(keys)} key(s)')

## 3. The Smart LLM Caller

Handles key rotation and rate-limit waiting. When a key is rate-limited, it:
1. Tries the next key for that layer (if available)
2. If all keys for the layer are limited, WAITS until the limit resets (doesn't skip)
3. Logs every call

In [ ]:
# Build a client for each unique key
CLIENTS = {key: Groq(api_key=key) for key in all_keys}

# Track which key index each layer is currently using
KEY_INDEX = {layer: 0 for layer in API_KEYS}

# Master log of all calls
CALL_LOG = []

def parse_retry_after(error_str):
    """Extract wait time from a Groq rate-limit error message."""
    # Groq errors often contain 'try again in 12.5s' or 'Please try again in 1m30s'
    m = re.search(r'try again in ([\dhms\.]+)', error_str, re.IGNORECASE)
    if m:
        t = m.group(1)
        seconds = 0
        h = re.search(r'(\d+)h', t)
        mm = re.search(r'(\d+)m', t)
        s = re.search(r'([\d\.]+)s', t)
        if h: seconds += int(h.group(1)) * 3600
        if mm: seconds += int(mm.group(1)) * 60
        if s: seconds += float(s.group(1))
        return max(seconds, 1)
    return 60  # default wait

def call_layer(layer_name, system_prompt, user_prompt, max_tokens=2048, temperature=0.5):
    """Call the LLM for a specific layer, with key rotation and rate-limit waiting."""
    keys = API_KEYS[layer_name]
    n_keys = len(keys)
    attempts = 0
    max_attempts = n_keys * 10  # generous retry budget
    
    while attempts < max_attempts:
        # Pick current key for this layer
        idx = KEY_INDEX[layer_name] % n_keys
        key = keys[idx]
        client = CLIENTS[key]
        
        try:
            resp = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {'role': 'system', 'content': system_prompt},
                    {'role': 'user', 'content': user_prompt}
                ],
                max_tokens=max_tokens,
                temperature=temperature
            )
            content = resp.choices[0].message.content
            
            # Log success
            CALL_LOG.append({
                'time': datetime.now().isoformat(),
                'layer': layer_name,
                'key_index': idx,
                'status': 'success',
                'tokens': max_tokens
            })
            return content
            
        except Exception as e:
            err = str(e)
            is_rate_limit = '429' in err or 'rate' in err.lower() or 'limit' in err.lower()
            
            CALL_LOG.append({
                'time': datetime.now().isoformat(),
                'layer': layer_name,
                'key_index': idx,
                'status': 'rate_limited' if is_rate_limit else 'error',
                'error': err[:150]
            })
            
            if is_rate_limit:
                # Try next key first
                if n_keys > 1:
                    KEY_INDEX[layer_name] = (idx + 1) % n_keys
                    # If we've cycled through all keys, wait
                    if (attempts + 1) % n_keys == 0:
                        wait = parse_retry_after(err)
                        print(f'    [{layer_name}] All {n_keys} keys limited. Waiting {wait:.0f}s for reset...')
                        time.sleep(wait)
                    else:
                        print(f'    [{layer_name}] Key {idx} limited, rotating to key {KEY_INDEX[layer_name]}')
                        time.sleep(1)
                else:
                    # Only one key - must wait
                    wait = parse_retry_after(err)
                    print(f'    [{layer_name}] Key limited. Waiting {wait:.0f}s for reset...')
                    time.sleep(wait)
            else:
                print(f'    [{layer_name}] Error: {err[:100]}')
                time.sleep(5)
            
            attempts += 1
    
    print(f'    [{layer_name}] Max attempts reached, giving up on this call')
    return None

def extract_json(text):
    """Extract JSON from LLM response."""
    if not text:
        return None
    text = text.strip()
    text = re.sub(r'^```(?:json)?\s*', '', text)
    text = re.sub(r'\s*```$', '', text)
    # Find first { or [
    candidates = [i for i in [text.find('{'), text.find('[')] if i >= 0]
    if candidates:
        text = text[min(candidates):]
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        # Try to find the last closing brace/bracket
        for end_char in ['}', ']']:
            last = text.rfind(end_char)
            if last > 0:
                try:
                    return json.loads(text[:last+1])
                except:
                    pass
        return None

print('Smart LLM caller ready!')

## 4. Layer Prompts

In [ ]:
LAYER_1A_SYSTEM = """You are a narrative analyst specializing in Islamic historical texts (Sira and Hadith).

Your task: Read a passage from the Prophet's biography and split it into DISTINCT NARRATIVE UNITS.
A narrative unit is one self-contained mini-story or event. A long passage often contains several.

Output a JSON array. Each unit has:
- unit_number: integer
- unit_title: short descriptive title
- unit_text: the relevant portion (paraphrase slightly for clarity if needed)
- main_characters: array of character names involved
- event_summary: one sentence describing what happens

Output ONLY the JSON array."""

LAYER_1B_SYSTEM = """You are an editorial director for a children's Islamic comic platform (ages 6-12).

Given a list of narrative units, decide which should become comic scenes.
CHOOSE units with clear visual moments, meaningful child-appropriate lessons, and distinctiveness.
DROP pure commentary, redundant units (keep only the most distinctive), or hard-to-illustrate abstractions.

Output a JSON array of SELECTED units. Each keeps its original fields plus:
- selection_reason: why chosen
- estimated_scenes: how many visual scenes (1-3)

Aim for 2-4 units from a long passage. Output ONLY the JSON array."""

LAYER_1C_SYSTEM = """You are a comic storyboard artist for children's Islamic education (ages 6-12).

Take ONE narrative unit and break it into individual VISUAL SCENES.
Each scene = exactly one moment that becomes one comic panel.

Output a JSON array of scenes. Each scene has:
- scene_number: integer
- scene_summary: what happens in this single moment
- key_visual_action: the specific action/gesture/pose
- characters_present: array of character names
- setting: where this takes place
- emotional_tone: the feeling
- time_of_day: morning/afternoon/evening/night

Output ONLY the JSON array."""

print('Layer 1 prompts loaded')

In [ ]:
LAYER_2_SYSTEM = """You are a specialist in creating children's Islamic educational comics (ages 6-12) for the Tariikhna platform.

Convert ONE visual scene into a complete schema with a DETAILED image generation prompt.

=== ISLAMIC DEPICTION RULES (CRITICAL) ===
- Prophets: NEVER show face. Show ONLY from behind, distinguished by clothing (e.g. green cloak). In the image_prompt, refer to a prophet as 'the central figure seen from behind' or 'the man in the green cloak' - NEVER use the literal word 'prophet' in image_prompt.
- Women: always fully and modestly dressed (jilbab, khimar covering hair). Describe clothing, not body.
- Others (companions, antagonists, crowds): shown normally with faces.
- No graphic violence. No idols in detail. No modern objects.

=== IMAGE PROMPT: MUST BE 250-300 WORDS ===
Include ALL of:
1. Art style: 'Children's watercolor storybook illustration, soft warm hand-painted style for ages 6-12'
2. Setting: specific 7th century location, architecture, furnishings, objects
3. Each character: build, posture, clothing with SPECIFIC colors, what they are doing
4. The specific action/gesture/emotion
5. Lighting: source, direction, quality, color
6. Composition: shot type, camera angle
7. Color palette: dominant and accent colors
8. End with: 'No modern objects, historically accurate 7th century scene.'

=== PHRASING TO AVOID (prevents image filter rejections) ===
- 'prophet' in image_prompt -> 'the central figure' / 'the man in the green cloak'
- 'young girl/child' + body -> 'young woman' or describe clothing only
- 'face hidden/obscured/dark' -> 'seen from behind' / 'turned away'
- 'baby/infant' + body -> 'a swaddled bundle'

=== OUTPUT (JSON only) ===
{
  "scene_title": "...",
  "era": "pre_islamic_prophets|ancient_prophets|pre_prophetic|early_makkah|late_makkah|madinah_early|madinah_late",
  "region": "makkah|madinah|arabian_desert|egypt_ancient|levant_ancient|sinai|abyssinia|other",
  "source_reference": "...",
  "moral_lesson": "...",
  "characters": [{"id":"...","name":"...","role":"prophet|sahabi|family_of_prophet|antagonist|supporting|crowd","depiction_rule":"NO_FACE|FROM_BEHIND|SILHOUETTE|FULL","appearance":"clothing and build, no facial features for prophets"}],
  "narrative_text": "1-3 simple child sentences",
  "compliance": {"prophet_check":"...","modesty_check":"COMPLIANT|NEEDS_REVIEW","child_safe":"APPROPRIATE|NEEDS_REVIEW","notes":"..."},
  "image_prompt": "the 250-300 word detailed prompt"
}
Output ONLY the JSON object."""

LAYER_2_EXAMPLE = """=== EXAMPLE OF EXPECTED QUALITY ===
Scene: {"scene_summary": "Abu Bakr weeps with joy when told he will be the companion", "key_visual_action": "hand pressed to chest, head bowed, tears", "characters_present": ["Prophet Muhammad", "Abu Bakr"], "setting": "Abu Bakr's home", "emotional_tone": "joy", "time_of_day": "afternoon"}

Output image_prompt (notice the LENGTH and DETAIL - this is your target):
"Children's watercolor storybook illustration, soft warm hand-painted style for ages 6-12. Interior of a modest 7th century Makkah home with unpainted mud-brick walls in warm earth tones and a packed-earth floor partly covered by a woven palm-fiber mat. Late afternoon golden sunlight streams through an open wooden doorway on the right, casting warm rays across the room. In the corner sits a tall clay water jug, and a small ceramic oil lamp rests on a low wooden shelf giving a soft glow. Two figures sit facing each other on the mat. On the left, the central figure seen entirely from behind: medium build, dignified upright posture, wearing a flowing white thobe with a distinctive deep green woolen cloak draped over both shoulders and a white turban with a cloth tail hanging between the shoulder blades, one hand extended in a gentle reassuring gesture. On the right, a slender older man with slightly stooped shoulders, wearing an off-white thobe and plain brown woolen cloak with a white cloth head covering, sitting in a deeply emotional posture - head tilted slightly forward, shoulders gently slumped, one hand pressed firmly to his chest, overcome with joyful tears. Soft warm golden light, intimate and sacred mood. Medium close shot at eye level. Warm sand, cream, and golden palette with the deep green cloak as a color anchor. No modern objects, historically accurate 7th century scene."
=== END EXAMPLE ==="""

LAYER_3_SYSTEM = """You are a content safety reviewer for an image generation pipeline.
Image models have oversensitive NSFW filters that wrongly reject innocent religious content.
Rewrite image prompts to avoid trigger words WITHOUT changing the scene or removing detail.

REWRITE:
- 'prophet' -> 'the central figure' / 'the man in the green cloak'
- 'young girl/child' + description -> 'young woman' or describe clothing
- 'face hidden/obscured/dark/covered' -> 'seen from behind' / 'head turned away'
- 'baby/infant' + body -> 'a small swaddled bundle'
- 'naked/bare' (even 'bare feet') -> 'sandaled feet' or omit

KEEP everything else identical. Do NOT shorten. Output JSON:
{"cleaned_prompt": "...", "changes_made": ["list of swaps"]}
Output ONLY the JSON."""

print('Layer 2 & 3 prompts loaded')

## 5. Layer Functions

In [ ]:
def save_log(passage_id, layer, raw_response):
    """Save a layer's raw response to a log file for review."""
    fname = f'pipeline_output/logs/{passage_id}_{layer}.txt'
    with open(fname, 'w', encoding='utf-8') as f:
        f.write(raw_response if raw_response else '[NO RESPONSE]')

def layer_1a_split(passage, passage_id):
    user = f"Split this passage into narrative units:\n\n{passage}"
    raw = call_layer('layer_1a', LAYER_1A_SYSTEM, user, max_tokens=2048, temperature=0.3)
    save_log(passage_id, '1a_units', raw)
    return extract_json(raw)

def layer_1b_select(units, passage_id):
    user = f"Select which units become comic scenes:\n\n{json.dumps(units, indent=2)}"
    raw = call_layer('layer_1b', LAYER_1B_SYSTEM, user, max_tokens=2048, temperature=0.3)
    save_log(passage_id, '1b_selected', raw)
    return extract_json(raw)

def layer_1c_segment(unit, passage_id, unit_idx):
    user = f"Break this unit into visual scenes:\n\n{json.dumps(unit, indent=2)}"
    raw = call_layer('layer_1c', LAYER_1C_SYSTEM, user, max_tokens=2048, temperature=0.4)
    save_log(passage_id, f'1c_scenes_unit{unit_idx}', raw)
    return extract_json(raw)

def layer_2_generate(scene, source_ref, passage_id, scene_idx):
    user = f"{LAYER_2_EXAMPLE}\n\nNow generate the schema for this scene:\n\n{json.dumps(scene, indent=2)}\n\nSource: {source_ref}"
    raw = call_layer('layer_2', LAYER_2_SYSTEM, user, max_tokens=2048, temperature=0.6)
    save_log(passage_id, f'2_schema_scene{scene_idx}', raw)
    return extract_json(raw)

def layer_3_filter(image_prompt, passage_id, scene_idx):
    user = f"Review and clean this image prompt:\n\n{image_prompt}"
    raw = call_layer('layer_3', LAYER_3_SYSTEM, user, max_tokens=1024, temperature=0.2)
    save_log(passage_id, f'3_filtered_scene{scene_idx}', raw)
    result = extract_json(raw)
    if result and 'cleaned_prompt' in result:
        return result['cleaned_prompt'], result.get('changes_made', [])
    return image_prompt, []

print('Layer functions ready!')

## 6. Full Pipeline (per passage)

In [ ]:
def process_passage(passage, source_ref, passage_id, verbose=True):
    """Run all layers on one passage. Returns full result with all intermediate data."""
    result = {
        'passage_id': passage_id,
        'passage': passage,
        'source_reference': source_ref,
        'units': None,
        'selected_units': None,
        'schemas': []
    }
    
    if verbose: print(f'  Layer 1a: splitting into units...')
    units = layer_1a_split(passage, passage_id)
    if not units:
        print('    FAILED at 1a')
        return result
    result['units'] = units
    if verbose: print(f'    {len(units)} units found')
    
    if verbose: print(f'  Layer 1b: selecting visual units...')
    selected = layer_1b_select(units, passage_id)
    if not selected:
        print('    FAILED at 1b')
        return result
    result['selected_units'] = selected
    if verbose: print(f'    {len(selected)} units selected')
    
    for u_idx, unit in enumerate(selected):
        if verbose: print(f'  Unit {u_idx+1}: {unit.get("unit_title", "?")}')
        scenes = layer_1c_segment(unit, passage_id, u_idx)
        if not scenes:
            if verbose: print(f'    segmentation failed, skipping')
            continue
        if verbose: print(f'    {len(scenes)} scenes')
        
        for s_idx, scene in enumerate(scenes):
            global_scene_id = f'u{u_idx}_s{s_idx}'
            schema = layer_2_generate(scene, source_ref, passage_id, global_scene_id)
            if not schema:
                if verbose: print(f'      scene {s_idx} failed at layer 2')
                continue
            
            # Layer 3 safety filter
            if 'image_prompt' in schema:
                cleaned, changes = layer_3_filter(schema['image_prompt'], passage_id, global_scene_id)
                schema['image_prompt'] = cleaned
                schema['_safety_changes'] = changes
            
            result['schemas'].append({'scene_input': scene, 'schema_output': schema})
            if verbose:
                wc = len(schema.get('image_prompt', '').split())
                print(f'      OK: {schema.get("scene_title", "?")} ({wc}w prompt)')
    
    return result

print('Pipeline ready!')

## 7. Input: Load Passages From a JSON File

Your input file should be a JSON array like:
```json
[
  {"passage_id": "abu_bakr_slaves", "passage": "full long text...", "source": "Adil Salahi Ch. X"},
  {"passage_id": "badr_battle", "passage": "...", "source": "..."}
]
```
Upload it to Colab, then set the path below.

In [ ]:
# ============================================
# SET YOUR INPUT FILE PATH
# ============================================
INPUT_FILE = 'my_passages.json'  # <- your file with long passages

# If you don't have a file yet, here's a demo with one passage:
DEMO_PASSAGES = [
    {
        'passage_id': 'abu_bakr_freeing_slaves',
        'passage': """Abu Bakr bought Amir ibn Fuhayrah, a slave among the very early Muslims who had suffered greatly, and set him free. Amir worked for Abu Bakr as a shepherd and later helped the Prophet and Abu Bakr emigrate to Madinah.

Zunayrah was a slave of the Makhzum clan, and Abu Jahl tortured her so severely she lost her sight. He claimed the gods al-Lat and al-Uzza had done it. She replied they could not know who worshipped them, and her Lord could restore her sight. The next day her sight returned. The Quraysh called it Muhammad's magic. Abu Bakr set her free.

Al-Nahdiyah and her daughter were slaves tortured by their mistress. One day the mistress gave them flour to bake and swore never to free them. Abu Bakr asked her to release herself from her oath, paid the price, and freed them.

Abu Bakr freed seven slaves in all. His father asked why he freed weak slaves instead of strong men who could protect him. Abu Bakr said he wanted only God's reward.""",
        'source': 'Adil Salahi, Muhammad: Man and Prophet, based on Ibn Hisham'
    }
]

# Try to load the file, fall back to demo
try:
    with open(INPUT_FILE) as f:
        passages = json.load(f)
    print(f'Loaded {len(passages)} passages from {INPUT_FILE}')
except FileNotFoundError:
    passages = DEMO_PASSAGES
    print(f'Input file not found - using {len(passages)} demo passage(s)')

# Show what we loaded
for p in passages[:3]:
    wc = len(p['passage'].split())
    print(f"  {p['passage_id']}: {wc} words")

## 8. Run the Full Batch (Resumable)

In [ ]:
def run_batch(passages, resume=True):
    """Process all passages. Saves after each. Resumable."""
    all_results = []
    
    # Resume: load existing results if present
    progress_file = 'pipeline_output/all_results.json'
    done_ids = set()
    if resume and os.path.exists(progress_file):
        with open(progress_file) as f:
            all_results = json.load(f)
        done_ids = {r['passage_id'] for r in all_results}
        print(f'Resuming: {len(done_ids)} passages already done')
    
    for i, p in enumerate(passages):
        pid = p['passage_id']
        if pid in done_ids:
            print(f'[{i+1}/{len(passages)}] {pid} - already done, skipping')
            continue
        
        print(f'\n[{i+1}/{len(passages)}] Processing: {pid}')
        res = process_passage(p['passage'], p.get('source', ''), pid, verbose=True)
        all_results.append(res)
        
        # Save individual schemas
        for j, item in enumerate(res['schemas']):
            schema_file = f'pipeline_output/schemas/{pid}_scene{j}.json'
            with open(schema_file, 'w', encoding='utf-8') as f:
                json.dump(item, f, ensure_ascii=False, indent=2)
        
        # Save progress
        with open(progress_file, 'w', encoding='utf-8') as f:
            json.dump(all_results, f, ensure_ascii=False, indent=2)
        
        # Save call log
        with open('pipeline_output/call_log.json', 'w', encoding='utf-8') as f:
            json.dump(CALL_LOG, f, indent=2)
        
        total_schemas = sum(len(r['schemas']) for r in all_results)
        print(f'  Saved. Total schemas so far: {total_schemas}')
    
    return all_results

# RUN IT
results = run_batch(passages, resume=True)
print(f'\n\nDONE! Processed {len(results)} passages.')
print(f'Total training schemas: {sum(len(r["schemas"]) for r in results)}')

## 9. Build JSONL Training File

In [ ]:
def to_training_example(scene_input, schema_output):
    clean = {k: v for k, v in schema_output.items() if not k.startswith('_')}
    return {
        'messages': [
            {'role': 'system', 'content': LAYER_2_SYSTEM},
            {'role': 'user', 'content': f"Generate the schema for this scene:\n\n{json.dumps(scene_input, indent=2)}"},
            {'role': 'assistant', 'content': json.dumps(clean, ensure_ascii=False)}
        ]
    }

training_examples = []
for res in results:
    for item in res['schemas']:
        training_examples.append(to_training_example(item['scene_input'], item['schema_output']))

with open('pipeline_output/tariikhna_training.jsonl', 'w', encoding='utf-8') as f:
    for ex in training_examples:
        f.write(json.dumps(ex, ensure_ascii=False) + '\n')

print(f'Saved {len(training_examples)} training examples to pipeline_output/tariikhna_training.jsonl')

## 10. Generate Visualizations for Your Graduation Document

Creates publication-ready charts from your pipeline run.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'DejaVu Sans'

# Gather statistics
n_passages = len(results)
n_units = sum(len(r['units']) for r in results if r['units'])
n_selected = sum(len(r['selected_units']) for r in results if r['selected_units'])
n_schemas = sum(len(r['schemas']) for r in results)

all_schemas = [item['schema_output'] for r in results for item in r['schemas']]
prompt_lengths = [len(s.get('image_prompt', '').split()) for s in all_schemas]
eras = [s.get('era', 'unknown') for s in all_schemas]
regions = [s.get('region', 'unknown') for s in all_schemas]
n_safety_changed = sum(1 for r in results for item in r['schemas'] if item['schema_output'].get('_safety_changes'))

print(f'Passages: {n_passages}')
print(f'Units: {n_units}, Selected: {n_selected}, Schemas: {n_schemas}')
print(f'Avg prompt length: {sum(prompt_lengths)/len(prompt_lengths):.0f} words' if prompt_lengths else 'no prompts')
print(f'Schemas safety-edited: {n_safety_changed}')

In [ ]:
# CHART 1: Data transformation funnel
fig, ax = plt.subplots(figsize=(10, 5))
stages = ['Raw\npassages', 'Narrative\nunits', 'Selected\nunits', 'Training\nscenes']
counts = [n_passages, n_units, n_selected, n_schemas]
colors = ['#534AB7', '#1D9E75', '#BA7517', '#D85A30']
bars = ax.barh(stages, counts, color=colors)
ax.set_xlabel('Count')
ax.set_title('Tariikhna Data Pipeline: Transformation Funnel', fontweight='bold')
for bar, c in zip(bars, counts):
    ax.text(bar.get_width() + max(counts)*0.01, bar.get_y() + bar.get_height()/2, str(c), va='center')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('pipeline_output/charts/01_funnel.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# CHART 2: Prompt length distribution (shows you hit the 250-300 target)
if prompt_lengths:
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.hist(prompt_lengths, bins=20, color='#1D9E75', edgecolor='white')
    ax.axvspan(250, 300, alpha=0.2, color='green', label='Target range (250-300)')
    ax.axvline(sum(prompt_lengths)/len(prompt_lengths), color='#D85A30', linestyle='--', label=f'Mean ({sum(prompt_lengths)/len(prompt_lengths):.0f}w)')
    ax.set_xlabel('Image prompt length (words)')
    ax.set_ylabel('Number of scenes')
    ax.set_title('Image Prompt Length Distribution', fontweight='bold')
    ax.legend()
    plt.tight_layout()
    plt.savefig('pipeline_output/charts/02_prompt_lengths.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# CHART 3: Era and region distribution
from collections import Counter
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

era_counts = Counter(eras)
ax1.pie(era_counts.values(), labels=era_counts.keys(), autopct='%1.0f%%',
        colors=plt.cm.Set3.colors)
ax1.set_title('Scenes by Era', fontweight='bold')

region_counts = Counter(regions)
ax2.bar(region_counts.keys(), region_counts.values(), color='#534AB7')
ax2.set_title('Scenes by Region', fontweight='bold')
ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('pipeline_output/charts/03_era_region.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# CHART 4: Safety filter impact
fig, ax = plt.subplots(figsize=(8, 5))
labels = ['Clean as generated', 'Required safety edit']
vals = [n_schemas - n_safety_changed, n_safety_changed]
ax.bar(labels, vals, color=['#1D9E75', '#BA7517'])
ax.set_ylabel('Number of scenes')
ax.set_title('Layer 3 Safety Filter Impact', fontweight='bold')
for i, v in enumerate(vals):
    ax.text(i, v + max(vals)*0.01, str(v), ha='center')
plt.tight_layout()
plt.savefig('pipeline_output/charts/04_safety.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nAll charts saved to pipeline_output/charts/')
print('Use these in your graduation document!')

## 11. Download Everything

Zip up all outputs for download.

In [ ]:
import shutil
shutil.make_archive('tariikhna_pipeline_output', 'zip', 'pipeline_output')
print('Created tariikhna_pipeline_output.zip')
print('\nContents:')
print('  - all_results.json (full pipeline data with units)')
print('  - schemas/ (individual schema JSONs)')
print('  - logs/ (raw response from every layer for review)')
print('  - tariikhna_training.jsonl (fine-tuning data)')
print('  - charts/ (visualizations for your document)')
print('  - call_log.json (API usage log)')

# In Colab, download with:
# from google.colab import files
# files.download('tariikhna_pipeline_output.zip')